# 饶舌歌生成：填歌词 → 一首带 beat 的饶舌（mp3）

只生成饶舌歌，不做视频。

1. 点菜单「代码执行程序 → 更改运行时类型」，选 **L4 GPU**，点「保存」。
2. 在下面「① 填歌词、选音色」填三项：歌词、音色、饶舌随机号（不改也能直接跑示范歌词）。
3. 点「代码执行程序 → 全部运行」（弹出「此笔记本不是由 Google 创作」时，点「仍然运行」）。跑完会出现播放条，并自动下载 mp3。

**用完一定要点「代码执行程序 → 断开连接并删除运行时」**，不然会一直扣计算单元。

In [ ]:
# @title ① 填歌词、选音色
# @markdown **歌词**：在这一格的代码里，把三个引号之间换成你的歌词，一行一句，引号要留着。每句 8～10 字最稳，字越多越容易吞字。歌长约＝字数×0.23 秒＋5 秒，最长 3 分半（210 秒），超过只会提醒、不会挡。
歌词 = """
周一早上闹钟响不停
挤上地铁人人没表情
老板开口又是新需求
咖啡续到第三杯不够
键盘敲到手指冒火星
下班时间还在改文件
走出大楼抬头看星星
明天照样准时去打拼
"""
# @markdown **音色**：「女·清亮」最接近作者试过的设定。音色是用文字提示 AI，效果有时明显有时不明显；不满意就换音色或换随机号再跑。
音色 = "女·清亮"  #@param ["女·清亮", "女·沙哑", "女·甜美", "女·御姐", "男·低沉", "男·少年", "男·烟嗓", "男·狠劲"]
# @markdown **饶舌随机号**：随机号像抽卡：同一份歌词换个号，节奏和念法就不同；没办法先知道哪个号好听，生成出来听，不喜欢就换个号再跑。
饶舌随机号 = 2025  #@param {type:"integer"}
# 表单栏位用中文名称显示；下面把值接回程序里用的变量名
LYRICS, TIMBRE, RAP_SEED = 歌词, 音色, 饶舌随机号
print(f"歌词 {len([s for s in LYRICS.splitlines() if s.strip()])} 句，音色：{TIMBRE}，饶舌随机号：{RAP_SEED}")

In [ ]:
# @title ② 生成饶舌 { display-mode: "form" }
# @markdown 不用改。生成完会出现播放条，并自动下载 mp3。换号、换音色再跑时，装好的工具和下载好的模型会接着用，不会重装。
# @markdown 作者估计（这一本还没在 Colab 上整本实测）：第一次约 2～4 分钟（要装工具、下载约 8GB 模型）；之后再跑约 1～2 分钟。歌越长，生成越久。这段时间画面不会动，是正常的。
import glob, html as _html, json, math, os, shutil, subprocess, sys, time
from IPython.display import Audio, HTML, display

SONG_DIR = "/content/rap_song"     # 成品放这里（左边文件面板看得到）
TOOL_DIR = "/root/rap_tools"       # 工具、模型、记录放这里（文件面板看不到；删除运行时才会清掉）
RAP_VENV = f"{TOOL_DIR}/venv"
RAP_CKPT = f"{TOOL_DIR}/acestep_ckpt"   # ACE-Step 主模型（约 8GB，第一次下载）
ACESTEP_COMMIT = "1bee4c9f5b43e30995f8d4d33b3919197ce1bd68"
RAP_PKGS = ["torch", "torchaudio", "torchvision", f"git+https://github.com/ace-step/ACE-Step.git@{ACESTEP_COMMIT}"]
RAP_PINS = ["torch==2.6.0", "torchaudio==2.6.0", "torchvision==0.21.0", "diffusers==0.33.1", "peft==0.15.2"]
RAP_LORA = "ACE-Step/ACE-Step-v1-chinese-rap-LoRA"
RAP_LORA_WEIGHT = 2.0   # LoRA 训练时的缩放是 2（r=256、alpha=32、rslora），pipeline 不读这个设置，所以手动给 2.0
RAP_PROMPT = ("Chinese Rap, battle rap, {g}, young adult, {g} rap, {tags}, spoken word, lyrical rap, aggressive, "
              "rhythmic spoken delivery, punchy flow, powerful, clear, boom bap, hard drums, 95 bpm")
# 音色 → (性别, 加进 prompt 的音色 tag, 文件名用的代号)。tag 取自 ACE-Step 官方 ZH_RAP_LORA.md 的 vocal_timbre／techniques 列表
# 和 examples/zh_rap_lora 的 prompt 范例（articulate、energetic、hardcore rap）
RAP_TIMBRES = {
    "女·清亮": ("female", "bright, articulate", "f_bright"),
    "女·沙哑": ("female", "husky, raspy, gritty", "f_husky"),
    "女·甜美": ("female", "light, airy, high-pitched", "f_sweet"),
    "女·御姐": ("female", "dark, sultry, velvety", "f_sultry"),
    "男·低沉": ("male", "dark, thick, resonant", "m_deep"),
    "男·少年": ("male", "bright, light, energetic", "m_young"),
    "男·烟嗓": ("male", "smoky, hoarse, raspy", "m_smoky"),
    "男·狠劲": ("male", "gritty, shouted, hardcore rap", "m_hard"),
}
RAP_SEC_PER_CHAR, RAP_PAD_SEC, RAP_MAX_SEC = 0.23, 5.0, 210   # 歌长＝字数×0.23 秒＋前奏和收尾约 5 秒，10～210 秒
# 生成用的子行程脚本（写成 /root/rap_tools/rap_gen.py，在独立环境里运行）
RAP_GEN_PY = r'''"""饶舌生成：ACE-Step v1 + 中文饶舌 LoRA（在独立的 Python 3.11 环境里运行，跑完行程结束就释放显存）。
用法：python rap_gen.py 设置.json   设置＝{ckpt, out, seed, prompt, lyrics, duration, lora, lora_weight}
参数照 2026-11-07 实测过的版本（官方 zh_rap_lora 范例＋lora_weight 2.0）。
"""
import glob
import json
import os
import shutil
import sys
import time

cfg = json.load(open(sys.argv[1], encoding="utf-8"))
t0 = time.time()
print("RAP_STEP 载入 ACE-Step（第一次要下载约 8GB 模型）", flush=True)
from acestep.pipeline_ace_step import ACEStepPipeline  # noqa: E402

p = ACEStepPipeline(checkpoint_dir=cfg["ckpt"], dtype="bfloat16")
p.load_checkpoint(p.checkpoint_dir)
print(f"RAP_STEP 模型就绪（{time.time() - t0:.0f} 秒），开始生成", flush=True)
t1 = time.time()
out_dir = os.path.dirname(cfg["out"])
os.makedirs(out_dir, exist_ok=True)
p(manual_seeds=[cfg["seed"]], lora_name_or_path=cfg["lora"], lora_weight=cfg["lora_weight"],
  save_path=cfg["out"], format="wav", audio_duration=cfg["duration"], prompt=cfg["prompt"], lyrics=cfg["lyrics"],
  infer_step=60, guidance_scale=15.0, scheduler_type="euler", cfg_type="apg", omega_scale=10.0,
  guidance_interval=0.5, guidance_interval_decay=0.0, min_guidance_scale=3.0, use_erg_tag=True,
  use_erg_lyric=False, use_erg_diffusion=True, oss_steps=None, guidance_scale_text=0.0,
  guidance_scale_lyric=0.0, batch_size=1, debug=True)
if not os.path.exists(cfg["out"]):   # 万一存档名和指定的不一样：取输出文件夹里最新的 wav
    wavs = sorted(glob.glob(os.path.join(out_dir, "*.wav")), key=os.path.getmtime)
    if not wavs:
        raise SystemExit("ACE-Step 跑完了，但没有找到生成的 wav")
    shutil.move(wavs[-1], cfg["out"])
print(f"RAP_STEP 生成完成（{time.time() - t1:.0f} 秒）", flush=True)
'''


class RapStop(Exception):
    """停下来（出错或没填好）：只显示白话说明，不显示一长串程序追踪。"""

    def _render_traceback_(self):
        return [str(self)]


def _rap_box(msg, kind):
    fg, bg = {"error": ("#8a1c1c", "#fde2e2"), "warn": ("#7a5200", "#fff3cd"), "info": ("#1d4f91", "#e3eefc"),
              "ok": ("#1e6b34", "#e2f3e6")}[kind]
    display(HTML(f'<div style="border-left:5px solid {fg};background:{bg};color:{fg};padding:6px 12px;'
                 f'margin:6px 0;white-space:pre-wrap">{_html.escape(str(msg))}</div>'))


def rap_fail(step, hint, err):
    """某一步失败：说清楚卡在哪、可以怎么做，细节印在下面给作者看。"""
    _rap_box(f"❌ 饶舌卡在「{step}」这一步。\n{hint}\n如果照做还是一样，请截图这一段（连同下面的细节）发给作者。", "error")
    print("—— 细节（给作者看）——\n" + "\n".join(str(err).splitlines()[-30:]))
    raise RapStop(f"饶舌卡在「{step}」：{hint}") from None


def _hint(err, default):
    s = str(err)
    if "out of memory" in s.lower() or "OutOfMemory" in s:
        return "显存不够。请点「代码执行程序 → 重启会话」，再点「全部运行」（装好的工具和下载好的模型会接着用）"
    if "No space left" in s:
        return "云端电脑的磁盘满了。请点「代码执行程序 → 断开连接并删除运行时」，重新连接后再点「全部运行」"
    return default


NET_HINT = "多半是网络临时不稳。请再点一次「代码执行程序 → 全部运行」，已经装好、下载好的部分会接着用"


def run(cmd, **kw):
    """运行外部命令；失败时打印最后 40 行输出并报错。"""
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        tail = "\n".join((r.stdout + r.stderr).splitlines()[-40:])
        raise RuntimeError(f"命令执行失败（错误代码 {r.returncode}）：{' '.join(map(str, cmd))}\n{tail}")
    return r


def build_venv(venv, pkgs, pins, label):
    """建独立的 Python 3.11 环境并装套件；装过（.ready 和这次要装的一样）就跳过。"""
    spec = json.dumps([pkgs, pins])
    try:
        with open(f"{venv}/.ready", encoding="utf-8") as f:
            if f.read() == spec:
                return
    except OSError:
        pass
    t = time.time()
    print(f"安装{label}（只有第一次要装，约 1～2 分钟）……", flush=True)
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    shutil.rmtree(venv, ignore_errors=True)
    run([sys.executable, "-m", "uv", "venv", "-q", "-p", "3.11", venv])
    cmd = [sys.executable, "-m", "uv", "pip", "install", "-q", "--python", f"{venv}/bin/python"]
    if pins:
        with open(f"{venv}/constraints.txt", "w") as f:
            f.write("\n".join(pins) + "\n")
        cmd += ["-c", f"{venv}/constraints.txt"]
    run(cmd + pkgs)
    with open(f"{venv}/.ready", "w", encoding="utf-8") as f:
        f.write(spec)
    print(f"  {label}装好了，耗时 {time.time() - t:.0f} 秒", flush=True)


def run_logged(cmd, log, env=None):
    """运行子行程，输出全部写进 log；印出 RAP_STEP 开头的进度；失败就报错（带最后 40 行）。"""
    with open(log, "w", encoding="utf-8") as f:
        rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env).returncode
    with open(log, encoding="utf-8", errors="replace") as f:
        lines = [ln.rstrip() for ln in f.read().replace("\r", "\n").splitlines() if ln.strip()]
    for ln in lines:
        if ln.startswith("RAP_"):
            print("  " + ln.split(" ", 1)[-1], flush=True)
    if rc:
        raise RuntimeError(f"错误代码 {rc}（完整记录：{log}）\n" + "\n".join(lines[-40:]))


def _rap_count(s):
    return sum(1 for c in s if c.isalnum())   # 念出来的字（中文字、英文字母、数字），不算标点和空格


def rap_plan(lyrics):
    """歌词 → (每句歌词, 总字数, 生成长度秒数, 提醒列表)。提醒只是提醒，不会挡下来。"""
    lines = [s.strip() for s in lyrics.strip().splitlines() if s.strip()]
    chars = sum(map(_rap_count, lines))
    want = math.ceil(chars * RAP_SEC_PER_CHAR + RAP_PAD_SEC)
    dur = max(10, min(RAP_MAX_SEC, want))
    warns = []
    long_lines = [f"第 {i} 句 {_rap_count(s)} 字" for i, s in enumerate(lines, 1) if _rap_count(s) > 12]
    if long_lines:
        warns.append("有几句比较长（" + "、".join(long_lines) + "）。每句 8～10 字最稳，字越多越容易吞字。只是提醒，照样会生成")
    if want > RAP_MAX_SEC:
        warns.append(f"歌词 {chars} 字，算出来要 {want} 秒，超过最长 3 分半（{RAP_MAX_SEC} 秒）：只会生成 {RAP_MAX_SEC} 秒，"
                     "后面的歌词多半唱不到。只是提醒，照样会生成")
    return lines, chars, dur, warns


# ---- 读 ① 的设置 ----
if "LYRICS" not in globals():
    raise RapStop("还没读到歌词：请先运行上面「① 填歌词、选音色」那一格（或直接点「代码执行程序 → 全部运行」）")
RAP_LINES, _chars, RAP_DUR, _warns = rap_plan(LYRICS)
if not RAP_LINES:
    raise RapStop("歌词是空的。请在 ① 的三个引号之间输入歌词（一行一句），再点「全部运行」")
if TIMBRE not in RAP_TIMBRES:
    raise RapStop(f"无法识别音色 {TIMBRE!r}：请在 ① 的「音色」下拉菜单里选一个")
_gender, _tags, _code = RAP_TIMBRES[TIMBRE]
SEED = int(RAP_SEED)

# ---- 检查 GPU（本项目只在 L4 上实测过）----
try:
    _smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
except OSError:
    _smi = ""
if not _smi:
    _rap_box("❌ 没有检测到 GPU。请点「代码执行程序 → 更改运行时类型」，选 L4 GPU，保存后再点「全部运行」", "error")
    raise RapStop("没有检测到 GPU：请改成 L4 GPU 再跑")
print(f"GPU：{_smi.splitlines()[0]}")
if "L4" not in _smi:
    _rap_box("⚠ 现在不是 L4。作者只在 L4 上试过，换别的 GPU 可能显存不够或很慢。\n"
             "建议点「代码执行程序 → 更改运行时类型」改成 L4。只是提醒，照样会跑", "warn")

print(f"音色：{TIMBRE}，饶舌随机号 {SEED}，歌词 {len(RAP_LINES)} 句、{_chars} 字 → 生成 {RAP_DUR} 秒")
for _w in _warns:
    _rap_box("⚠ " + _w, "warn")

# ---- 装工具 → 生成 ----
_rap_box("⏳ 生成中（第一次要装工具、下载约 8GB 模型，比较久）。这段时间画面不会动，是正常的", "warn")
t0 = time.time()
WORK = f"{TOOL_DIR}/work"
os.makedirs(SONG_DIR, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
for _old in glob.glob(f"{WORK}/*.wav"):   # 清掉上一次的暂存，免得拿错文件
    os.remove(_old)
with open(f"{TOOL_DIR}/rap_gen.py", "w", encoding="utf-8") as f:
    f.write(RAP_GEN_PY)
try:
    build_venv(RAP_VENV, RAP_PKGS, RAP_PINS, "饶舌工具（ACE-Step）")
except Exception as e:
    shutil.rmtree(RAP_VENV, ignore_errors=True)   # 装到一半的环境删掉，下次从头装
    rap_fail("安装饶舌工具", _hint(e, NET_HINT), e)
_cfg = {"ckpt": RAP_CKPT, "out": f"{WORK}/gen.wav", "seed": SEED, "duration": float(RAP_DUR),
        "prompt": RAP_PROMPT.format(g=_gender, tags=_tags), "lyrics": "[verse]\n" + "\n".join(RAP_LINES),
        "lora": RAP_LORA, "lora_weight": RAP_LORA_WEIGHT}
with open(f"{WORK}/gen_config.json", "w", encoding="utf-8") as f:
    json.dump(_cfg, f, ensure_ascii=False, indent=1)
try:
    print("生成饶舌（ACE-Step）……", flush=True)
    run_logged([f"{RAP_VENV}/bin/python", f"{TOOL_DIR}/rap_gen.py", f"{WORK}/gen_config.json"], f"{WORK}/gen_log.txt")
except Exception as e:
    rap_fail("下载模型／生成饶舌", _hint(e, "多半是下载模型时网络不稳。" + NET_HINT), e)

# ---- 存成品：wav 留在 /content/rap_song/，另转一份 320k 的 mp3 ----
_name, _k = f"rap_{SEED}_{_code}", 1
while os.path.exists(f"{SONG_DIR}/{_name}.wav") or os.path.exists(f"{SONG_DIR}/{_name}.mp3"):   # 同号重跑不盖掉旧的
    _k += 1
    _name = f"rap_{SEED}_{_code}_{_k}"
SONG_WAV, SONG_MP3 = f"{SONG_DIR}/{_name}.wav", f"{SONG_DIR}/{_name}.mp3"
try:
    shutil.move(f"{WORK}/gen.wav", SONG_WAV)
    run(["ffmpeg", "-v", "error", "-y", "-i", SONG_WAV, "-codec:a", "libmp3lame", "-b:a", "320k", SONG_MP3])
    _len = float(run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0",
                      SONG_MP3]).stdout.strip())
except Exception as e:
    rap_fail("转成 mp3", "请再点一次「代码执行程序 → 全部运行」", e)

print(f"完成：{_name}.mp3，{_len:.1f} 秒，本次耗时 {time.time() - t0:.0f} 秒")
display(Audio(SONG_MP3))
try:
    from google.colab import files
    files.download(SONG_MP3)
except Exception as e:
    print(f"自动下载没成功（{e!r}）")
print("如果没自动下载：左边文件面板 → rap_song → 右键下载")
_rap_box(f"✅ 生成完成：{_name}.mp3（{_len:.1f} 秒）。可以在上面试听。\n"
         "不喜欢：在 ① 换个随机号（或换音色），再点「代码执行程序 → 全部运行」，装好的工具和模型会接着用。\n"
         "用完记得点「代码执行程序 → 断开连接并删除运行时」", "ok")

## 跑完之后（一定要做）
- mp3 会自动下载到浏览器的下载文件夹。没下载的话：左边文件面板 → `rap_song` → 右键下载（同一个文件夹里也有无损的 wav）。
- 不喜欢这一版：回 ① 换个随机号或音色，再点「全部运行」。
- **最后一定要点「代码执行程序 → 断开连接并删除运行时」。** 不删的话，闲置时也会一直扣计算单元。第一次点可能会先弹出警告，确认后要再点一次删除。